# Self-Hosted Therapist Models on Colab (Free GPU)

Use this notebook for the two therapist models that have **no free hosted API**:

| Model | HF repo | Base | License |
|---|---|---|---|
| MentaLLaMA-chat-7B | `klyang/MentaLLaMA-chat-7B` | LLaMA-2-7B-chat | MIT |
| MentaLLaMA-chat-13B | `klyang/MentaLLaMA-chat-13B` | LLaMA-2-13B-chat | MIT |
| ChatCounselor ("ChatPsychiatrist") | `EmoCareAI/ChatPsychiatrist` | LLaMA-7B | Apache-2.0 |

Both are ungated on the Hub (no Meta license click-through needed) and load
fine with 4-bit quantization on a **free Colab T4 GPU (15GB VRAM)**.

**Runtime setup:** `Runtime -> Change runtime type -> T4 GPU` before running.

**How this fits the rest of the project:** the framework itself (patient/
therapist/evaluator agents, prompts, loop controller, analytics) runs the
same in Colab as it does locally — the only Colab-specific piece is
*loading the therapist model with a GPU*. The patient and evaluator agents
keep hitting their free APIs (Gemini/Mistral/Groq) over the internet, which
Colab has access to, so everything runs in this one notebook end-to-end.

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q transformers accelerate bitsandbytes sentencepiece google-genai mistralai groq python-dotenv pandas matplotlib

## 1. Upload the project code

Zip the `psych_llm_benchmark/` project folder you were given and upload it
here (or upload it to Google Drive and unzip from there so it persists
across sessions).

In [ ]:
from google.colab import files
uploaded = files.upload()  # select psych_llm_benchmark.zip

import zipfile, os
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/')

# adjust this if your zip extracts to a differently-named folder
PROJECT_DIR = '/content/psych_llm_benchmark'
os.chdir(PROJECT_DIR)
import sys
sys.path.insert(0, PROJECT_DIR)
print('Project loaded at', PROJECT_DIR)

## 2. Set API keys for the Patient + Evaluator agents

These stay on free hosted APIs even while the Therapist model runs locally
on the Colab GPU. Get free keys at:
- Gemini: https://aistudio.google.com/apikey
- Mistral: https://console.mistral.ai/api-keys
- Groq: https://console.groq.com/keys

Only set the keys for the providers you plan to use (e.g. if using Groq for
patient + evaluator, you can skip Gemini and Mistral).

In [ ]:
import os
from getpass import getpass

# Set the keys for the providers you're using for patient + evaluator.
# Comment out any you don't need.
os.environ['GEMINI_API_KEY'] = getpass('Gemini API key: ')
os.environ['MISTRAL_API_KEY'] = getpass('Mistral API key: ')
os.environ['GROQ_API_KEY'] = getpass('Groq API key: ')

## 3. Load the self-hosted therapist model in 4-bit

Pick ONE of the two model IDs below by uncommenting it.

In [ ]:
MODEL_ID = 'klyang/MentaLLaMA-chat-7B'
# MODEL_ID = 'EmoCareAI/ChatPsychiatrist'
# MODEL_ID = 'klyang/MentaLLaMA-chat-13B'   # needs Colab Pro / A100 for comfortable headroom

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f'Loading {MODEL_ID} in 4-bit... this can take a few minutes on first run.')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
)
print('Model loaded. GPU memory:')
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

In [ ]:
from agents.llm_clients import HFLocalClient, get_client

# Reuses the already-loaded model/tokenizer (no reload cost) — same interface
# as every API-based client, so it drops straight into TherapistAgent.
therapist_llm = HFLocalClient(model_name=MODEL_ID, model=model, tokenizer=tokenizer)

# --- Patient agent (pick ONE) ---
patient_llm = get_client('gemini', 'gemini-2.5-flash')
# patient_llm = get_client('mistral', 'mistral-large-latest')
# patient_llm = get_client('groq', 'llama-3.1-8b-instant')

# --- Evaluator agent (pick ONE — use a different model from the therapist) ---
evaluator_llm = get_client('mistral', 'mistral-large-latest')
# evaluator_llm = get_client('gemini', 'gemini-2.5-flash')
# evaluator_llm = get_client('groq', 'llama-3.3-70b-versatile')

## 4. Run the benchmark

Pick which of the 9 conditions to run against this model. Each session is
10 exchanges by default — start with 1-2 conditions to sanity-check output
quality/latency before running all 9.

In [ ]:
from analytics.exporter import AnalyticsExporter
from run_benchmark import run_single_session

conditions_to_run = ['BPD', 'Schizophrenia', 'MDD']  # edit as needed, or use list(PATIENT_PROFILES.keys()) for all 9
exporter = AnalyticsExporter()

for condition in conditions_to_run:
    run_single_session(condition, patient_llm, therapist_llm, evaluator_llm,
                        max_turns=10, exporter=exporter)

## 5. Build charts + qualitative 'why' report for this model

In [ ]:
from analytics.aggregator import load_all_evaluations, load_full_records
from analytics.visualize import generate_all_charts
from analytics.why_analysis import WhyAnalyzer

df = load_all_evaluations()
chart_paths = generate_all_charts(df)
print(chart_paths)

records = load_full_records()
why_analyzer = WhyAnalyzer(llm=evaluator_llm)
report_paths = why_analyzer.analyze_all(records)
print(report_paths)

## 6. Download the logs (transcripts, evaluations, charts, why-reports)

Colab runtimes are ephemeral — zip the `logs/` folder and download it (or
copy to Drive) before your session disconnects.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('/content/logs_export', 'zip', os.path.join(PROJECT_DIR, 'logs'))
files.download('/content/logs_export.zip')